# שיעור 32 — פתרון מתועד: פונקציות ומימוש עצמאי של KNN
3 שעות לימוד מוצעות (135 דקות),במסגרת תכנית 90 השעות המעודכנת.

**הקשר ללמידת מכונה:** דודו יודע להפעיל ספרייה. כעת יוכל להסביר ולבדוק כיצד מרחק,מיון והצבעה הופכים לתחזית,בלי להסתיר אותם בקריאת predict.

**עוגן:** [סיכום KNN מפגש 5](../../07-KNN/Meeting-05-Didactic-Summary.md),שיעור 23 ו[מדריך המימוש ב־DL](../../../03-Deep-Learning-5-Units/07-Classification-KNN/Meeting-05-KNN-Implementation-Guide.md). זו גרסה בלולאות לפני Broadcasting,לא העתק מחברת המרצה.

המחברת היא התאמה דידקטית חדשה. אין קובץ מקור חדש מהמרצה ולכן אין Exact-Original מלאכותי. אין לייחס לה אישור כיסוי מחוון עדכני.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
X_train=np.array([[2.,1.],[4.,2.],[4.,4.],[6.,2.],[2.,7.]])
y_train=np.array(["B","A","A","B","A"])
queries=np.array([[2.,2.]])

## נתונים ופונקציית מרחק
הנתונים מומצאים,באותו סולם וללא השלמה/תקנון; אין כאן מבחן Generalization.

In [ ]:
def euclidean_distance(u,v):
    total=0.0
    for j in range(len(u)):
        total=total+(float(u[j])-float(v[j]))**2
    return total**0.5
print("Distance check:",euclidean_distance([1,2],[4,6]))

## סיווג יחיד
שומרים לכל מרחק תווית מתאימה. K חייב להיות שלם בין 1 למספר דוגמאות האימון; כאן הערכים נתונים ותקינים.

In [ ]:
def predict_one(train_X,train_y,query,k):
    distances=[]
    for row in train_X:
        distances.append(euclidean_distance(row,query))
    neighbors=pd.DataFrame({"distance":distances,"label":train_y})
    nearest=neighbors.sort_values("distance",kind="stable").head(k)
    return nearest["label"].mode().iloc[0]
manual_predictions=[]
for q in queries:
    manual_predictions.append(predict_one(X_train,y_train,q,3))
library=KNeighborsClassifier(n_neighbors=3,weights="uniform",metric="euclidean")
library.fit(X_train,y_train)
print("Manual:",manual_predictions,"Library:",library.predict(queries))

## מתאם מחלקה נתון
האלגוריתם נשאר בפונקציות; self שומר אימון ו־k. fit אינו יוצר עמודת מרחק קבועה לכל שאילתה.

In [ ]:
class SimpleKNN:
    def __init__(self,k=3):
        self.k=k
    def fit(self,X,y):
        self.X=np.array(X,dtype=float).copy()
        self.y=np.array(y).copy()
        return self
    def predict(self,X):
        output=[]
        for q in X:
            output.append(predict_one(self.X,self.y,q,self.k))
        return np.array(output)
custom=SimpleKNN(k=3)
custom.fit(X_train,y_train)
print("Class interface:",custom.predict(queries))
# אותה תוצאה כווקטור מרחקים; הרחבה אחרי הבנת הלולאה
vector_distances=np.sqrt(((X_train-queries[0])**2).sum(axis=1))
print(vector_distances)

## פירוש ומגבלה
לא שוכחים return; לא ממיינים מרחקים בלי התוויות; לא מצביעים בכל השורות במקום K; לא מייחסים Accuracy לשאילתה שאין לה אמת. המתאם אינו מחלקת sklearn מלאה ואינו נבדק ב־GridSearchCV. נדרש קלט דו־ממדי,אימון לא ריק וממדים תואמים; אין טיפול כללי בשגיאות בתרגיל.
המספרים נקראים מפלט ההרצה; אין לדרוש ציון ביצועים מסוים. הסבר חלופי מנומק יכול להיות נכון.